# WaRek Advisor API Test

Test endpoint `/api/advisor` dengan data transaksi real untuk memverifikasi AI insights bekerja dengan baik.

## Setup: Pastikan backend berjalan di http://localhost:8000

In [ ]:
import requests
import json
from datetime import datetime, timedelta

API_BASE = 'http://localhost:8000'

# Test health endpoint dulu
resp = requests.get(f'{API_BASE}/health')
print(f'Health check: {resp.status_code}')
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

## Scenario 1: Warung Normal (Seimbang)

Pemasukan dan pengeluaran seimbang, kategori yang terdefinisi baik.

In [ ]:
# Generate sample transactions untuk 30 hari terakhir
transactions_1 = [
    {"tanggal": (datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d'), 
     "keterangan": "Penjualan nasi goreng", 
     "jumlah": 150000, 
     "tipe": "pemasukan", 
     "kategori": "penjualan"}
    for i in range(0, 30, 2)
]

transactions_1.extend([
    {"tanggal": (datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d'), 
     "keterangan": "Beli bahan baku", 
     "jumlah": 80000, 
     "tipe": "pengeluaran", 
     "kategori": "bahan_baku"}
    for i in range(0, 30, 2)
])

transactions_1.extend([
    {"tanggal": (datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d'), 
     "keterangan": "Gaji karyawan", 
     "jumlah": 200000, 
     "tipe": "pengeluaran", 
     "kategori": "gaji"}
    for i in range(0, 30, 7)
])

# Sort by tanggal descending
transactions_1.sort(key=lambda x: x['tanggal'], reverse=True)

print(f"Total transactions: {len(transactions_1)}")
print(f"Sample (3 recent):")
for tx in transactions_1[:3]:
    print(json.dumps(tx, indent=2, ensure_ascii=False))

In [ ]:
# Test advisor endpoint
resp = requests.post(
    f'{API_BASE}/api/advisor',
    json={'transactions': transactions_1},
    headers={'Content-Type': 'application/json'}
)

print(f'Status: {resp.status_code}')
if resp.status_code == 200:
    data = resp.json()
    insight = data.get('data')
    print(f"\n📊 AI INSIGHT SCENARIO 1 (Normal):")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"Insight: {insight.get('insight')}")
    print(f"Tipe: {insight.get('tipe')}")
    print(f"Aksi: {insight.get('aksi')}")
else:
    print(f'Error: {resp.text}')

## Scenario 2: Warning - Pengeluaran Tinggi

Pengeluaran mendominasi, cashflow negatif.

In [ ]:
# Scenario dengan pengeluaran tinggi
transactions_2 = [
    {"tanggal": (datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d'), 
     "keterangan": "Penjualan nasi goreng", 
     "jumlah": 100000, 
     "tipe": "pemasukan", 
     "kategori": "penjualan"}
    for i in range(0, 30, 3)
]

transactions_2.extend([
    {"tanggal": (datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d'), 
     "keterangan": "Beli bahan baku banyak", 
     "jumlah": 200000, 
     "tipe": "pengeluaran", 
     "kategori": "bahan_baku"}
    for i in range(0, 30, 2)
])

transactions_2.extend([
    {"tanggal": (datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d'), 
     "keterangan": "Bayar listrik", 
     "jumlah": 150000, 
     "tipe": "pengeluaran", 
     "kategori": "operasional"}
    for i in range(0, 30, 5)
])

transactions_2.sort(key=lambda x: x['tanggal'], reverse=True)

resp = requests.post(
    f'{API_BASE}/api/advisor',
    json={'transactions': transactions_2},
    headers={'Content-Type': 'application/json'}
)

print(f'Status: {resp.status_code}')
if resp.status_code == 200:
    data = resp.json()
    insight = data.get('data')
    print(f"\n⚠️  AI INSIGHT SCENARIO 2 (High Expense):")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"Insight: {insight.get('insight')}")
    print(f"Tipe: {insight.get('tipe')}")
    print(f"Aksi: {insight.get('aksi')}")
else:
    print(f'Error: {resp.text}')

## Scenario 3: Positive - Pertumbuhan Penjualan

Pemasukan terus meningkat, cashflow positif konsisten.

In [ ]:
# Scenario dengan pemasukan tinggi
transactions_3 = [
    {"tanggal": (datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d'), 
     "keterangan": "Penjualan nasi goreng", 
     "jumlah": int(150000 + i * 5000),  # Increasing trend
     "tipe": "pemasukan", 
     "kategori": "penjualan"}
    for i in range(0, 30, 1)
]

transactions_3.extend([
    {"tanggal": (datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d'), 
     "keterangan": "Beli bahan baku", 
     "jumlah": 80000, 
     "tipe": "pengeluaran", 
     "kategori": "bahan_baku"}
    for i in range(0, 30, 2)
])

transactions_3.sort(key=lambda x: x['tanggal'], reverse=True)

resp = requests.post(
    f'{API_BASE}/api/advisor',
    json={'transactions': transactions_3},
    headers={'Content-Type': 'application/json'}
)

print(f'Status: {resp.status_code}')
if resp.status_code == 200:
    data = resp.json()
    insight = data.get('data')
    print(f"\n✅ AI INSIGHT SCENARIO 3 (Growth):")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"Insight: {insight.get('insight')}")
    print(f"Tipe: {insight.get('tipe')}")
    print(f"Aksi: {insight.get('aksi')}")
else:
    print(f'Error: {resp.text}')

## Summary

✅ Test selesai. Advisor AI memberikan:
- **Insight** yang relevan sesuai kondisi keuangan
- **Tipe** feedback (warning/tip/positive)
- **Aksi** yang bisa langsung diambil user

Semua dalam bahasa Indonesia santai yang mudah dipahami pemilik warung.